In [ ]:
!pip install -U huggingface_hub
!pip install -q pyngrok

In [ ]:
!wget -O llama-server-cuda.tar.gz "https://github.com/iqzaardiansyah/llama-cuda-builds/releases/download/git-661643e43/llama-server-git-661643e43-ubuntu22.04-x86_64-cu128-sm75-py3.12.tar.gz"
!tar -xzvf llama-server-cuda.tar.gz -C /kaggle/working/

In [ ]:
!hf download unsloth/Qwen3.8-27B-GGUF --include "*Q4_K_M.gguf" --local-dir ./models
!hf download unsloth/Qwen3.8-27B-GGUF --include "MTP/mtp-Qwen3.8-27B-Q4_0.gguf" --local-dir ./models

In [ ]:
import os
import subprocess
import glob
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
ngrok.set_auth_token(user_secrets.get_secret("NGROK_AUTH_TOKEN"))

for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
    
public_url = ngrok.connect(8080).public_url
print("=" * 60)
print(f"OpenAI-Compatible API URL: {public_url}/v1")
print("=" * 60)

working_dir = "/kaggle/working"
so_dirs = set(os.path.dirname(p) for p in glob.glob(f"{working_dir}/**/*.so*", recursive=True))
custom_ld_path = ":".join(so_dirs)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = custom_ld_path + ":" + env.get("LD_LIBRARY_PATH", "")

server_cmd = [
    f"{working_dir}/bin/llama-server", 
    "--model", f"{working_dir}/models/Qwen3.8-27B-UD-Q4_K_M.gguf",
    "--model-draft", f"{working_dir}/models/MTP/mtp-Qwen3.8-27B-Q4_0.gguf", 
    "--spec-draft-n-max", "3",
    "--host", "0.0.0.0",
    "--port", "8080",
    "-ngl", "-1",               
    "-fa", "on",                      
    "-ctk", "q8_0",             
    "-ctv", "q8_0",             
    "-b", "512",
    "-ub", "512",
    "--ctx-size", "65536",
    "-np", "4",
    "-cb"
]

print("Starting server...")
subprocess.run(server_cmd, env=env)